In [1]:
from db_connector import *

========| ('irfan_admin', 'trade_app') |========
*** ✅ SUCCESSFUL CLOUD CONNECTION ⛓️ ***


In [2]:
df = fn_read_data_cloud("gold","bist_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","ams_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","nasdaq_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","nyse_master_combined_indicators")

In [ ]:
df = fn_read_data_cloud("gold","crypto_master_combined_indicators")

In [ ]:
# copy_log_bist_master_combined_indicators
# df = fn_read_data_cloud("test","copy_log_bist_master_combined_indicators")
df = fn_read_data_cloud("test","copy_log_nasdaq_master_combined_indicators")


In [ ]:
df

In [3]:
def calc_poc_cluster_score(df):

    def cluster(group):
        n = group["FRVP_HIGHEST_DATE"].nunique()
        vals = group["FRVP_POC"].values

        if n == 1:
            return 3

        spread = (vals.max() - vals.min()) / vals.min()

        if n >= 3:
            if spread == 0:
                return 10
            elif spread <= 0.03:
                return 3 + (1 - spread / 0.03) * 7
            else:
                return 3

        if n == 2:
            if spread == 0:
                return 7
            elif spread <= 0.03:
                return 3 + (1 - spread / 0.03) * 4
            else:
                return 3

        return 0

    # 🔥 DOĞRU YÖNTEM
    cluster_scores = (
        df.groupby(["EXCHANGE", "SYMBOL"])
        .apply(cluster)
        .rename("cluster_score")
        .reset_index()
    )

    # merge ile geri bağla
    df = df.merge(cluster_scores, on=["EXCHANGE", "SYMBOL"], how="left")

    return df["cluster_score"]

In [4]:
def calc_poc_cluster_bonus(df):
    def cluster_bonus(g):
        g_unique = g.drop_duplicates(subset=["FRVP_HIGHEST_DATE"]).copy()
        count = len(g_unique)

        if count < 2:
            return 0

        pocs = g_unique["FRVP_POC"].dropna().astype(float).values
        if len(pocs) < 2:
            return 0

        spread = (pocs.max() - pocs.min()) / pocs.min()

        if spread <= 0.02:
            if count >= 4:
                return 15
            elif count == 3:
                return 10
            elif count == 2:
                return 5

        return 0

    bonus_df = (
        df.groupby(["EXCHANGE", "SYMBOL"], group_keys=False)
        .apply(lambda g: pd.Series({"score7": cluster_bonus(g)}))
        .reset_index()
    )

    out = df.merge(bonus_df, on=["EXCHANGE", "SYMBOL"], how="left")
    return out["score7"].fillna(0).to_numpy()

In [5]:
import numpy as np
import pandas as pd

# =========================
# HELPERS
# =========================
def to_float(col):
    return (
        col.astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# =========================
# MFI
# =========================
def calc_mfi_new(df):
    return (
        np.where(df["MFI"] > df["MFI_YESTERDAY"], 25, 0) +
        np.where(df["MFI"] > df["MFI_12DAY_AVG"], 50, 0) +
        np.where(df["MFI_DIRECTION"] == "Upward", 25, 0)
    )

# =========================
# RSI
# =========================
def calc_rsi_new(df):
    rsi = to_float(df["RSI"])
    rsi_ma = pd.to_numeric(df["RSI_MA"], errors="coerce")
    status = pd.to_numeric(df["RSI_STATUS"], errors="coerce").fillna(0)
    days = pd.to_numeric(df["RSI_CROSS_DAYS_AGO"], errors="coerce").fillna(0)

    score1 = np.where(rsi > rsi_ma, 70, 0)

    days = df["RSI_CROSS_DAYS_AGO"]
    score2 = np.where(
        (df["RSI_STATUS"] == 1) & (days <= 14),
        np.maximum(0, 15 - days),
        0
    )

    score3 = np.where((rsi >= 30) & (rsi <= 70), 15, 0)

    return np.clip(score1 + score2 + score3, 0, 100)

# =========================
# EMA
# =========================
def calc_ema_new(df):
    def ema_score(status, cross, days):
        status = pd.to_numeric(status, errors="coerce").fillna(0)
        cross  = pd.to_numeric(cross, errors="coerce").fillna(0)
        days   = pd.to_numeric(days, errors="coerce").fillna(0)

        return np.where(
            (status == 1) & (cross == 1),
            np.maximum(0, 3 - (days / 10)),
            0
        )

    total = (
        ema_score(df["EMA_STATUS_5_20"], df["EMA_CROSS_5_20"], df["EMA_DAYS_SINCE_CROSS_5_20"]) +
        ema_score(df["EMA_STATUS_3_20"], df["EMA_CROSS_3_20"], df["EMA_DAYS_SINCE_CROSS_3_20"]) +
        ema_score(df["EMA_STATUS_3_14"], df["EMA_CROSS_3_14"], df["EMA_DAYS_SINCE_CROSS_3_14"])
    )

    return np.clip((total / 9) * 100, 0, 100)

# =========================
# VOLUME
# =========================
def calc_volume_new(df):
    vol_last = to_float(df["VOL_LASTDAY"])
    vol_yest = to_float(df["VOL_YESTERDAY"])
    vol_5 = to_float(df["VOL_AVG_5DAY"])
    vol_10 = to_float(df["VOL_AVG_10DAY"])
    vol_20 = to_float(df["VOL_AVG_20DAY"])

    return (
        np.where(vol_last > vol_yest, 25, 0) +
        np.where(vol_last > vol_5, 25, 0) +
        np.where(vol_5 > vol_10, 25, 0) +
        np.where(vol_5 > vol_20, 25, 0)
    )

# =========================
# VWAP
# =========================
def calc_vwap_new(df):
    close = df["FRVP_LATEST_CLOSE_VALUE"]
    vwap = df["VWAP"]

    dist = np.abs(close - vwap) / vwap

    return np.clip(
        np.where(dist <= 0.05, 0,
                 np.where(dist >= 0.10, 100,
                          ((dist - 0.05) / 0.05) * 100)),
        0, 100
    )

# =========================
# POC FRVP
# =========================

def calc_poc_frvp_new(df):

    close = df["FRVP_LATEST_CLOSE_VALUE"]
    poc = df["FRVP_POC"]
    val = df["FRVP_VAL"]
    vah = df["FRVP_VAH"]

    # 1
    score1 = np.clip((1 - ((close - poc) / poc) / 0.20) * 5, 0, 5)

    # 2 (cluster FIXED)
    score2 = calc_poc_cluster_score(df)

    # 3 ✅ GREEN FIX
    score3 = np.where(df["BS_BAR_STATUS"] == "GREEN", 5, 0)

    # 4
    score4 = np.where(
        (df["BS_OPEN_PRICE"] > poc) &
        (df["BS_BAR_STATUS"] == "GREEN"),
        5,
        0
    )

    # 5
    score5 = np.clip((1 - np.abs(poc - val) / val / 0.20) * 5, 0, 5)

    # 6
    dist_vah = np.abs(poc - vah) / vah
    score6 = np.where(
        dist_vah >= 0.10, 5,
        np.where(
            dist_vah <= 0.05, 0,
            ((dist_vah - 0.05) / 0.05) * 5
        )
    )

    # 7 (yeni bonus)
    score7 = calc_poc_cluster_bonus(df)

    total = score1 + score2 + score3 + score4 + score5 + score6 + score7

    return np.clip((total / 50) * 100, 0, 100)

# =========================
# TRADE LEVELS
# =========================

def calculate_trade_levels(df):

    close = df["FRVP_LATEST_CLOSE_VALUE"]
    df["entry_price"] = close * 1.005

    pivot = df["PIVOT"] if "PIVOT" in df.columns else pd.Series(np.nan, index=df.index)
    r2 = df["R2"] if "R2" in df.columns else pd.Series(np.nan, index=df.index)

    def choose_target(row):

        c = row["FRVP_LATEST_CLOSE_VALUE"]

        if c < row["FRVP_POC"]:
            return row["VWAP"]

        elif c < row["VWAP"]:
            return row["VWAP"]

        elif c < row["FRVP_VAH"]:
            return row["FRVP_VAH"]

        elif pd.notna(pivot.loc[row.name]) and c < pivot.loc[row.name]:
            return pivot.loc[row.name]

        elif pd.notna(r2.loc[row.name]):
            return r2.loc[row.name]

        else:
            return row["FRVP_VAH"]

    df["target_price"] = df.apply(choose_target, axis=1)

    df["target_pct"] = ((df["target_price"] - df["entry_price"]) / df["entry_price"]) * 100
    df["risk_pct"] = ((df["entry_price"] - df["stop_loss"]) / df["entry_price"]) * 100
    df["rr_ratio"] = df["target_pct"] / df["risk_pct"]

    df["pivot_display"] = np.where(
        pivot.isna(),
        "N/A",
        pivot
    )

    return df

# =========================
# MASTER FUNCTION
# =========================

def calculate_scores(df):

    df["poc_frvp_status"] = calc_poc_frvp_new(df)
    df["vwap_status"] = calc_vwap_new(df)
    df["ema_status"] = calc_ema_new(df)
    df["rsi_status"] = calc_rsi_new(df)
    df["mfi_status"] = calc_mfi_new(df)
    df["vol_status"] = calc_volume_new(df)

    # =========================
    # MASTER SCORE
    # =========================
    df["master_score"] = (
        df["poc_frvp_status"] * 0.65 +
        df["vwap_status"] * 0.02 +
        df["ema_status"] * 0.09 +
        df["rsi_status"] * 0.07 +
        df["mfi_status"] * 0.07 +
        df["vol_status"] * 0.10
    )

    df["master_score"] = np.clip(df["master_score"], 0, 100)

    # =========================
    # WATCHLIST
    # =========================
    df["watchlist"] = np.where(
        df["master_score"] >= 70, "BUY",
        np.where(df["master_score"] >= 50, "WATCH", "AVOID")
    )
    df["entry_price"] = df["FRVP_LATEST_CLOSE_VALUE"] * 1.005
    

    # =========================
    # STOP LOSS (ENTRY’den sonra!)
    # =========================
    df["stop_loss"] = df["entry_price"] * 0.97

    # =========================
    # TRADE LEVELS (ENTRY burada oluşuyor)
    # =========================
    df = calculate_trade_levels(df)

    return df

In [6]:
scores = calculate_scores(df)

In [7]:
print(scores.to_clipboard(index=False))

None


In [ ]:
scores

In [ ]:
df.head(5)


# --- TRIAGE KISMI----

In [10]:
def calculate_triage_selection(df):

    # df_filtered = df[df["BS_CLOSE_PRICE"] > df["FRVP_POC"]].copy()
    df_filtered = df[
    (df["BS_CLOSE_PRICE"] > df["FRVP_POC"])].copy()

    if df_filtered.empty:
        return pd.DataFrame()

    grouped = df_filtered.groupby(["EXCHANGE", "SYMBOL"])

    results = []

    for (exchange, symbol), g in grouped:

        # =========================
        # UNIQUE CLUSTER COUNT
        # =========================
        g_unique = g.drop_duplicates(subset=["FRVP_HIGHEST_DATE"])
        count = len(g_unique)

        # =========================
        # SCORE
        # =========================
        avg_score = g_unique["master_score"].mean()

        # =========================
        # POC CLUSTER ANALYSIS
        # =========================
        pocs = g_unique["FRVP_POC"].values

        poc_counts = g["FRVP_POC"].value_counts()

        max_freq = poc_counts.max()

        dominant_pocs = poc_counts[poc_counts == max_freq].index

        avg_poc = np.mean(dominant_pocs)

        # =========================
        # TRADE LEVELS (UPDATED)
        # =========================

        last_close = g["BS_CLOSE_PRICE"].iloc[-1]

        # STOP LOSS → %3 aşağı
        stop_loss_price = last_close * (1 - 0.03)

        # TARGET → mevcut avg_target yerine relative hesap
        avg_target = g["target_price"].mean()

        target_pct = ((avg_target - last_close) / last_close) * 100

        triage_day = pd.to_datetime(g["CREATED_DAY"].iloc[0], dayfirst=True).strftime("%Y-%m-%d")

        results.append({
            "EXCHANGE": exchange,
            "SYMBOL": symbol,
            "triage_entry_day": triage_day,

            "triage_score": round(min(100, avg_score), 2),
            "valid_cluster_count": count,

            "avg_poc": f"{avg_poc:.2f}".replace(".", ","),

            # ✅ yeni yapı
            "stop_loss": f"{stop_loss_price:.2f}".replace(".", ","),
            "target_price": f"{avg_target:.2f}".replace(".", ","),

            # ✅ yüzde artık dinamik
            "target_%": f"{target_pct:.2f}%".replace(".", ",")
        })

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values("triage_score", ascending=False)

    return result_df


BACK TEST (SADECE YASIN ICIN)

In [ ]:
scores

In [11]:
df = calculate_triage_selection(scores)

In [12]:
print(df.to_clipboard(index=False))

None


In [ ]:
df.head(10)

In [ ]:
# hourly ilgili olani al
# df_hourly = fn_read_data_cloud("raw","bist_hourly_archive")
df_hourly = fn_read_data_cloud("raw","nasdaq_hourly_archive")

In [ ]:
import pandas as pd
import numpy as np


RESULT_COLUMNS = [
    "expected_entry_price",
    "entry_time",
    "calculated_stop_loss",
    "close_position_day",
    "position_duration",
    "close_position_price",
    "return_%",
    "max_price_seen",
    "max_price_day",
    "max_return_%",
    "exit_reason",
]


def _to_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", ".", regex=False)
         .str.replace("%", "", regex=False)
         .str.strip(),
        errors="coerce"
    )


def prepare_signals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()

    required_cols = ["SYMBOL", "triage_entry_day", "target_price"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Signal dataframe missing columns: {missing}")

    df["SYMBOL"] = df["SYMBOL"].astype(str).str.strip()
    df["triage_entry_day"] = pd.to_datetime(df["triage_entry_day"], errors="coerce")
    df["target_price"] = _to_float_series(df["target_price"])

    return df


def prepare_hourly(df_hourly: pd.DataFrame) -> pd.DataFrame:
    dfh = df_hourly.copy()
    dfh.columns = dfh.columns.str.strip()

    required_cols = ["SYMBOL", "TIMESTAMP", "OPEN", "HIGH", "LOW", "CLOSE"]
    missing = [c for c in required_cols if c not in dfh.columns]
    if missing:
        raise ValueError(f"Hourly dataframe missing columns: {missing}")

    dfh["SYMBOL"] = dfh["SYMBOL"].astype(str).str.strip()
    dfh["TIMESTAMP"] = pd.to_datetime(dfh["TIMESTAMP"], errors="coerce")

    for col in ["OPEN", "HIGH", "LOW", "CLOSE"]:
        dfh[col] = _to_float_series(dfh[col])

    dfh = dfh.dropna(subset=["SYMBOL", "TIMESTAMP", "OPEN", "HIGH", "LOW", "CLOSE"]).copy()
    dfh["date"] = dfh["TIMESTAMP"].dt.date
    dfh["hour"] = dfh["TIMESTAMP"].dt.hour

    dfh = dfh.sort_values(["SYMBOL", "TIMESTAMP"]).reset_index(drop=True)
    return dfh


def empty_result(reason=np.nan) -> dict:
    return {
        "expected_entry_price": np.nan,
        "entry_time": pd.NaT,
        "calculated_stop_loss": np.nan,
        "close_position_day": pd.NaT,
        "position_duration": np.nan,
        "close_position_price": np.nan,
        "return_%": np.nan,
        "max_price_seen": np.nan,
        "max_price_day": pd.NaT,
        "max_return_%": np.nan,
        "exit_reason": reason,
    }


def evaluate_one_position(
    row: pd.Series,
    hourly_by_symbol: dict,
    trading_days_by_symbol: dict,
    stop_loss_pct: float = 0.03,
) -> dict:
    symbol = row["SYMBOL"]
    signal_day = row["triage_entry_day"]
    target = row["target_price"]

    if pd.isna(signal_day) or pd.isna(target):
        return empty_result("missing_signal_data")

    signal_day = signal_day.date()

    sym_df = hourly_by_symbol.get(symbol)
    sym_days = trading_days_by_symbol.get(symbol)

    if sym_df is None or sym_df.empty or not sym_days:
        return empty_result("missing_hourly_data")

    future_days = [d for d in sym_days if d > signal_day]
    if not future_days:
        return empty_result("no_future_trading_day")

    # Day+1
    t1 = future_days[0]

    # Day+1 barları
    day_df = sym_df.loc[sym_df["date"] == t1].sort_values("TIMESTAMP").reset_index(drop=True)
    if len(day_df) < 2:
        return empty_result("insufficient_day1_bars")

    first_candle = day_df.iloc[0]
    second_candle = day_df.iloc[1]

    # 2. mum yeşil değilse trade yok
    if second_candle["CLOSE"] <= second_candle["OPEN"]:
        return empty_result("entry_candle_not_green")

    # Entry
    entry_price = float(second_candle["CLOSE"])
    entry_time = second_candle["TIMESTAMP"]

    # Stop loss artık entry'den türetiliyor
    stop_loss = entry_price * (1 - stop_loss_pct)

    # Long sanity
    if target <= entry_price:
        return empty_result("invalid_target_price")

    # Day+1 dahil sonraki 7 borsa günü
    allowed_days = future_days[:7]

    df_future = sym_df.loc[
        (sym_df["TIMESTAMP"] > entry_time) &
        (sym_df["date"].isin(allowed_days))
    ].sort_values("TIMESTAMP").reset_index(drop=True)

    if df_future.empty:
        return {
            "expected_entry_price": round(entry_price, 4),
            "entry_time": entry_time,
            "calculated_stop_loss": round(stop_loss, 4),
            "close_position_day": pd.NaT,
            "position_duration": np.nan,
            "close_position_price": np.nan,
            "return_%": np.nan,
            "max_price_seen": np.nan,
            "max_price_day": pd.NaT,
            "max_return_%": np.nan,
            "exit_reason": "no_future_bars",
        }

    close_time = None
    close_price = None
    exit_reason = None

    for _, bar in df_future.iterrows():
        low = float(bar["LOW"])
        high = float(bar["HIGH"])
        ts = bar["TIMESTAMP"]

        # Aynı mumda hem target hem stop varsa stop öncelikli
        if low <= stop_loss:
            close_time = ts
            close_price = stop_loss
            exit_reason = "stop_loss"
            break

        if high >= target:
            close_time = ts
            close_price = target
            exit_reason = "target"
            break

    if close_time is None:
        last_bar = df_future.iloc[-1]
        close_time = last_bar["TIMESTAMP"]
        close_price = float(last_bar["CLOSE"])
        exit_reason = "timeout_close"

    # Exit anına kadar görülen max high
    df_until_exit = df_future.loc[df_future["TIMESTAMP"] <= close_time]

    if not df_until_exit.empty:
        max_idx = df_until_exit["HIGH"].idxmax()
        max_price = float(df_until_exit.loc[max_idx, "HIGH"])
        max_time = df_until_exit.loc[max_idx, "TIMESTAMP"]
    else:
        max_price = np.nan
        max_time = pd.NaT

    # Target ile çıkıldıysa max en az exit fiyatı kadar olmalı
    if exit_reason == "target" and pd.notna(max_price) and max_price < close_price:
        max_price = close_price
        max_time = close_time

    duration_days = (close_time.date() - entry_time.date()).days if pd.notna(close_time) else np.nan
    return_pct = ((close_price / entry_price) - 1.0) * 100 if pd.notna(close_price) else np.nan
    max_return_pct = ((max_price / entry_price) - 1.0) * 100 if pd.notna(max_price) else np.nan

    return {
        "expected_entry_price": round(entry_price, 4),
        "entry_time": entry_time,
        "calculated_stop_loss": round(stop_loss, 4),
        "close_position_day": close_time,
        "position_duration": f"{duration_days} days" if pd.notna(duration_days) else np.nan,
        "close_position_price": round(close_price, 4) if pd.notna(close_price) else np.nan,
        "return_%": round(return_pct, 2) if pd.notna(return_pct) else np.nan,
        "max_price_seen": round(max_price, 4) if pd.notna(max_price) else np.nan,
        "max_price_day": max_time,
        "max_return_%": round(max_return_pct, 2) if pd.notna(max_return_pct) else np.nan,
        "exit_reason": exit_reason,
    }


def build_hourly_cache(df_hourly: pd.DataFrame):
    hourly_by_symbol = {}
    trading_days_by_symbol = {}

    for symbol, g in df_hourly.groupby("SYMBOL", sort=False):
        gs = g.sort_values("TIMESTAMP").reset_index(drop=True).copy()
        hourly_by_symbol[symbol] = gs
        trading_days_by_symbol[symbol] = sorted(gs["date"].unique().tolist())

    return hourly_by_symbol, trading_days_by_symbol


def run_backtest(
    signals_df: pd.DataFrame,
    hourly_df: pd.DataFrame,
    stop_loss_pct: float = 0.03,
) -> pd.DataFrame:
    signals = prepare_signals(signals_df)
    hourly = prepare_hourly(hourly_df)

    hourly_by_symbol, trading_days_by_symbol = build_hourly_cache(hourly)

    results = [
        evaluate_one_position(
            row,
            hourly_by_symbol,
            trading_days_by_symbol,
            stop_loss_pct=stop_loss_pct,
        )
        for _, row in signals.iterrows()
    ]

    results_df = pd.DataFrame(results, index=signals.index)
    final_df = pd.concat([signals, results_df], axis=1)

    ordered_cols = list(signals.columns) + RESULT_COLUMNS
    ordered_cols = [c for c in ordered_cols if c in final_df.columns]

    return final_df[ordered_cols]

In [ ]:
df_final = run_backtest(df, df_hourly)

In [ ]:
print(df_final.to_clipboard(index=False))

In [ ]:
def get_daily_top2_positions(df_final):
    
    df = df_final.copy()
    
    # sadece gerçekten açılan pozisyonlar
    df = df[df["entry_time"].notna()]
    
    # tarih normalize (datetime ise)
    df["triage_entry_day"] = pd.to_datetime(df["triage_entry_day"])
    
    # sıralama (istersen score'a göre de değiştirebiliriz)
    df = df.sort_values(["triage_entry_day", "entry_time"])
    
    # her gün için ilk 2
    df_top2 = (
        df.groupby("triage_entry_day")
        .head(2)
        .reset_index(drop=True)
    )
    
    return df_top2

In [ ]:
df_top2 = get_daily_top2_positions(df_final)

In [ ]:
print(df_top2.to_clipboard(index=False))